In [2]:
#             monthly_summary.csv
#             category_breakdown.csv
#             utilities_trend.csv
#             grocery_trend.csv
#             income_vs_expense.csv


In [3]:
import pandas as pd

path = 's3://ds-learning-hunny/aws-practice-data/banking/data/processed/bank_transactions_cleaned.csv'

df = pd.read_csv(path)
df.head()

c:\Users\hunny\AppData\Local\Programs\Python\Python311\Lib\site-packages\fsspec\registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,month,date,transaction_type,description,category,amount,balance
0,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84


In [4]:
print("-------------------------------------------- Monthly Summary ---------------------------------------------------")
total_income = df[df['category'] == 'Income'].groupby('month')['amount'].sum().reset_index().rename(columns={'amount':'total_income'})
total_income

-------------------------------------------- Monthly Summary ---------------------------------------------------


,month,total_income
0,2026-01,2450.0
1,2026-02,2450.0
2,2026-03,2450.0


In [5]:
df_no_income = df[df['transaction_type'] == 'withdrawal']
total_expense = df_no_income.groupby('month')['amount'].sum().abs().reset_index().rename(columns={'amount':'total_expense'})
total_expense

,month,total_expense
0,2026-01,1486.26
1,2026-02,1621.87
2,2026-03,1541.73


In [6]:
monthly_summary = total_income.merge(total_expense, on='month')
monthly_summary['net_saving'] = (monthly_summary['total_income']-monthly_summary['total_expense'])

monthly_summary

,month,total_income,total_expense,net_saving
0,2026-01,2450.0,1486.26,963.74
1,2026-02,2450.0,1621.87,828.13
2,2026-03,2450.0,1541.73,908.27


In [7]:
# Saving as CSV to local and AWS s3 Folder
monthly_summary.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/analytics/monthly_summary.csv',index=False)
monthly_summary.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/analytics/monthly_summary.csv',index=False)


In [8]:
print("-------------------------------------------- Category breakdown ---------------------------------------------------")
pivot  = df_no_income.groupby(['month','category'])['amount'].sum().reset_index().rename(columns={'amount':'total_spent'})


monthly_category_spend = pivot.pivot_table(
    index='month',
    columns='category',
    values='total_spent',
    fill_value=0
)

monthly_category_spend.columns.name = None
monthly_category_spend['total_expense'] = monthly_category_spend.sum(axis=1)
monthly_category_spend


-------------------------------------------- Category breakdown ---------------------------------------------------


,Entertainment,Food,Groceries,Housing,Insurance,Shopping,Transportation,Utilities,total_expense
month,,,,,,,,,
2026-01,-15.49,0.0,-74.22,-1200.0,0.0,0.00,-42.1,-154.45,-1486.26
2026-02,0.00,-62.4,0.00,-1200.0,-135.0,-53.18,0.0,-171.29,-1621.87
2026-03,-10.99,0.0,-142.89,-1200.0,0.0,0.00,-64.5,-123.35,-1541.73


In [9]:
# Saving as CSV to local and AWS s3 Folder
monthly_category_spend.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/analytics/category_breakdown.csv')
monthly_category_spend.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/analytics/category_breakdown.csv')


In [10]:
print("-------------------------------------------- utilities trend ---------------------------------------------------")
# utilities_total by month
utilities_total = monthly_category_spend['Utilities']
utilities_total=utilities_total.reset_index()

-------------------------------------------- utilities trend ---------------------------------------------------


In [11]:
# Saving as CSV to local and AWS s3 Folder
utilities_total.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/analytics/utilities_trend.csv', index=False)
utilities_total.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/analytics/utilities_trend.csv',index=False)

In [12]:
print("-------------------------------------------- grocer trend ---------------------------------------------------")
#  grocer_trend by month
grocer_trend = monthly_category_spend['Groceries']
grocer_trend= grocer_trend.reset_index()
grocer_trend

-------------------------------------------- grocer trend ---------------------------------------------------


,month,Groceries
0,2026-01,-74.22
1,2026-02,0.00
2,2026-03,-142.89


In [13]:
# Saving as CSV to local and AWS s3 Folder
grocer_trend.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/analytics/grocer_trend.csv', index=False)
grocer_trend.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/analytics/grocer_trend.csv',index=False)

In [24]:
print("-------------------------------------------- income_vs_expense ---------------------------------------------------")
# income_vs_expense by month
# categories: date, income, expense. 
df.head()

-------------------------------------------- income_vs_expense ---------------------------------------------------


,month,date,transaction_type,description,category,amount,balance
0,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84


In [82]:
income_vs_expense = df[['month','date']].reset_index().drop(columns={'index'})
total_income_withdate  = df[df['category']=='Income'].drop(columns={'transaction_type','description','category','month'})

total_expense_by_day = df[df['category'] != 'Income'].groupby('date')['amount'].sum()
total_expense_by_day = total_expense_by_day.reset_index()

income_vs_expense = income_vs_expense.merge(total_income_withdate[['date','amount']],on='date',how='left').rename(columns={'amount':'Income'})
income_vs_expense = income_vs_expense.merge(total_expense_by_day,on='date',how='left').rename(columns={'amount' : 'expense'})

income_vs_expense['Income'] = income_vs_expense['Income'].fillna(0)
income_vs_expense['expense'] = income_vs_expense['expense'].fillna(0)


income_vs_expense = income_vs_expense.merge(df[['date','balance']],on='date',how='left')

income_vs_expense.head()


,month,date,Income,expense,balance
0,2026-01,2026-01-02,2450.0,0.00,2450.00
1,2026-01,2026-01-03,0.0,-1200.00,1250.00
2,2026-01,2026-01-05,0.0,-86.45,1163.55
3,2026-01,2026-01-08,0.0,-74.22,1089.33
4,2026-01,2026-01-12,0.0,-15.49,1073.84


In [83]:
# Saving as CSV to local and AWS s3 Folder
income_vs_expense.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/analytics/income_vs_expense.csv', index=False)
income_vs_expense.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/analytics/income_vs_expense.csv',index=False)